## dbNSFP 5.3.1a extraction

Streams a 48 GB archive and keeps only our 197,904 variants. Nothing large touches disk; each chromosome is checkpointed to Drive so a disconnect costs one chromosome, not the run.

Measured against this server: sustained streams run ~2.6 MB/s, ranged reads ~1.4, and parallel connections are *slower* than one. So each chromosome is pulled as a single sustained connection. Expect roughly five hours, resumable across sessions.

**Your download URL contains a personal access code — do not save or share this notebook with it filled in.**

In [ ]:
!pip -q install pyarrow requests
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = '/content/drive/MyDrive/dbnsfp_extract'
os.makedirs(OUT, exist_ok=True)
print('checkpoints ->', OUT)

Upload `data/rebuild/dbnsfp_join_keys.csv.gz` (1.7 MB). It carries join keys only — no pathogenic/benign labels leave your machine.

In [ ]:
# Upload data/rebuild/dbnsfp_join_keys.csv.gz (1.7 MB) when prompted.
import pandas as pd

if not os.path.exists(f'{OUT}/dbnsfp_join_keys.csv.gz'):
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    with open(f'{OUT}/dbnsfp_join_keys.csv.gz', 'wb') as f:
        f.write(up[name])

keys = pd.read_csv(f'{OUT}/dbnsfp_join_keys.csv.gz')
print(f'{len(keys):,} variants, {keys.uniprot_acc.nunique():,} accessions')

Paste your `dbNSFP5.3.1a.zip` URL below. This cell parses the zip directory to find where each chromosome lives, then reads the header to discover which predictors this release ships.

In [ ]:
import requests, io, struct, gzip, time

URL = ''  # @param {type:"string"}  <- paste your dbNSFP5.3.1a.zip URL here
assert URL.endswith('.zip'), 'paste the dbNSFP5.3.1a.zip URL'

SESSION = requests.Session()
_h = SESSION.head(URL, allow_redirects=True); _h.raise_for_status()
SIZE = int(_h.headers['Content-Length'])
print(f'archive {SIZE/1e9:.1f} GB')

def rng(start, end):
    """One small ranged read. Used only for parsing the zip directory."""
    for attempt in range(5):
        try:
            r = SESSION.get(URL, headers={'Range': f'bytes={start}-{end}'},
                            timeout=180)
            r.raise_for_status()
            return r.content
        except Exception as e:
            if attempt == 4:
                raise
            print(f'  retry {attempt+1} after {type(e).__name__}', flush=True)

# ---- end-of-central-directory, then the directory itself -------------------
tail = rng(max(0, SIZE - 66000), SIZE - 1)
i = tail.rfind(b'PK\x05\x06')
cd_size, cd_off = struct.unpack('<II', tail[i+12:i+20])
if cd_off == 0xFFFFFFFF or cd_size == 0xFFFFFFFF:          # ZIP64
    j = tail.rfind(b'PK\x06\x06')
    cd_size, cd_off = struct.unpack('<QQ', tail[j+40:j+56])
# a directory read has a known sane size; a bad parse here would otherwise
# request gigabytes and look like a hang rather than an error
assert cd_size < 50_000_000, f'implausible central directory size {cd_size}'
assert 0 < cd_off < SIZE, f'implausible central directory offset {cd_off}'
cd = rng(cd_off, cd_off + cd_size - 1)

def zip64(extra, usize, csize, hoff):
    """Resolve fields that overflowed their 32-bit slots.

    In an archive this size every local-header offset, and any member above
    4 GB, is stored as 0xFFFFFFFF with the real value in the ZIP64 extra field
    (header id 0x0001). The 8-byte values appear in a fixed order - uncompressed
    size, compressed size, local header offset - but ONLY for the fields that
    actually overflowed, so they have to be consumed conditionally.
    """
    p = 0
    while p + 4 <= len(extra):
        hid, hsz = struct.unpack('<HH', extra[p:p+4])
        body = extra[p+4:p+4+hsz]
        if hid == 0x0001:
            q = 0
            if usize == 0xFFFFFFFF:
                usize, = struct.unpack('<Q', body[q:q+8]); q += 8
            if csize == 0xFFFFFFFF:
                csize, = struct.unpack('<Q', body[q:q+8]); q += 8
            if hoff == 0xFFFFFFFF:
                hoff, = struct.unpack('<Q', body[q:q+8]); q += 8
            break
        p += 4 + hsz
    return usize, csize, hoff


MEMBERS, p = [], 0
while p + 46 <= len(cd) and cd[p:p+4] == b'PK\x01\x02':
    method, = struct.unpack('<H', cd[p+10:p+12])
    csize, usize = struct.unpack('<II', cd[p+20:p+28])
    nlen, elen, clen = struct.unpack('<HHH', cd[p+28:p+34])
    hoff, = struct.unpack('<I', cd[p+42:p+46])
    name = cd[p+46:p+46+nlen].decode('utf8', 'replace')
    extra = cd[p+46+nlen:p+46+nlen+elen]
    usize, csize, hoff = zip64(extra, usize, csize, hoff)
    if 'variant' in name and name.endswith('.gz'):
        MEMBERS.append({'name': name, 'method': method, 'csize': csize,
                        'hoff': hoff})
    p += 46 + nlen + elen + clen
MEMBERS.sort(key=lambda m: m['name'])
print(f'{len(MEMBERS)} chromosome members, '
      f'{sum(m["csize"] for m in MEMBERS)/1e9:.1f} GB compressed')

# ---- each member's data offset, from its local header ----------------------
# The local header's extra field length differs from the central directory's,
# so the data offset has to be read from the local header itself.
for m in MEMBERS:
    assert m['method'] == 0, f"{m['name']} is deflated, not stored"
    assert 0 < m['hoff'] < SIZE, f"{m['name']}: bad offset {m['hoff']}"
    assert 0 < m['csize'] < SIZE, f"{m['name']}: bad size {m['csize']}"
    lh = rng(m['hoff'], m['hoff'] + 29)
    assert lh[:4] == b'PK\x03\x04', f"{m['name']}: no local header at offset"
    nlen, elen = struct.unpack('<HH', lh[26:30])
    m['start'] = m['hoff'] + 30 + nlen + elen
    m['end'] = m['start'] + m['csize'] - 1
    assert m['end'] < SIZE, f"{m['name']}: runs past end of archive"
print('all member offsets resolved and checked')

class MemberReader(io.RawIOBase):
    """Exactly csize bytes of one zip member, with a clean EOF.

    The response is a ranged stream covering precisely this member. Handing the
    raw urllib3 stream to BufferedReader means end-of-stream arrives as a
    closed-file exception rather than b'', which surfaces deep inside gzip as
    'read of closed file'. Counting bytes gives a proper EOF, and records
    whether the stream was truncated so a dropped connection can be told apart
    from a member that simply ended.
    """
    def __init__(self, resp, nbytes):
        self.resp, self.raw, self.left = resp, resp.raw, nbytes
        self.truncated = False

    def readable(self):
        return True

    def readinto(self, b):
        if self.left <= 0:
            return 0
        try:
            chunk = self.raw.read(min(len(b), self.left))
        except (ValueError, OSError):
            chunk = b''
        if not chunk:
            self.truncated = self.left > 0
            self.left = 0
            return 0
        b[:len(chunk)] = chunk
        self.left -= len(chunk)
        return len(chunk)


def member_stream(m, attempts=6):
    """One sustained connection for a whole member, verified before use.

    Two failures have to be caught here rather than deep inside gzip: the server
    intermittently answers 200 with the whole archive instead of 206 with the
    requested range, and a ranged response can start at the wrong offset. Both
    show up as a gzip magic-number error thousands of lines later, so the
    response is checked at the point it is created.
    """
    last = None
    for a in range(attempts):
        r = SESSION.get(URL, headers={'Range': f"bytes={m['start']}-{m['end']}"},
                        stream=True, timeout=900)
        try:
            r.raise_for_status()
            if r.status_code != 206:
                last = f'status {r.status_code}, range ignored'
                r.close(); time.sleep(3 * (a + 1)); continue
            cr = r.headers.get('Content-Range', '')
            got = int(cr.split()[1].split('-')[0]) if cr else -1
            if got != m['start']:
                last = f'Content-Range starts at {got}, wanted {m["start"]}'
                r.close(); time.sleep(3 * (a + 1)); continue

            r.raw.decode_content = False
            buf = io.BufferedReader(MemberReader(r, m['csize']),
                                    buffer_size=16 << 20)
            magic = buf.peek(2)[:2]          # peek does not consume
            if magic != b'\x1f\x8b':
                last = f'stream starts {magic!r}, not gzip'
                r.close(); time.sleep(3 * (a + 1)); continue
            return buf, r
        except Exception as e:
            last = f'{type(e).__name__}: {e}'
            try:
                r.close()
            except Exception:
                pass
            time.sleep(3 * (a + 1))
        print(f'  retry {a+1} for {m["name"]}: {last}', flush=True)
    raise RuntimeError(f'{m["name"]}: {last}')

# ---- column discovery, from the first member -------------------------------
_buf, _r = member_stream(MEMBERS[0])
_head = gzip.GzipFile(fileobj=_buf).readline().decode('utf8', 'replace')
_r.close()
ALL = _head.rstrip('\n').split('\t')

KEYCOLS = [c for c in ['#chr', 'pos(1-based)', 'ref', 'alt', 'genename',
                       'Uniprot_acc', 'aapos', 'aaref', 'aaalt'] if c in ALL]
# rankscore columns are normalised twins of the raw scores and would
# double-count in any correlation across predictors; _pred columns are
# categorical calls, not scores.
SCORES = [c for c in ALL if c.endswith('_score')
          and not c.endswith('_rankscore') and 'converted' not in c]
print(f'{len(ALL)} columns in this release, {len(SCORES)} score columns kept')
for c in SCORES[:8]:
    print('  ', c)
print('   ...')

The scan. Re-run this cell after any disconnect — finished chromosomes are skipped.

In [ ]:
import time, os, glob, json, numpy as np

USE = KEYCOLS + SCORES
CHUNK = 250_000
REPORT_EVERY = 2        # chunks between progress lines
PART_EVERY = 10         # chunks between partial checkpoints

# ---- match keys as frames, so matching is a merge not per-row lookups ------
KEYDF = keys.rename(columns={'uniprot_acc': 'acc', 'position_1': 'pos'})[
    ['acc', 'gene_symbol', 'pos', 'wt_aa', 'mut_aa']].copy()
KEYDF['pos'] = KEYDF['pos'].astype('int64')
KEYACC = KEYDF[['acc', 'pos', 'wt_aa', 'mut_aa']].drop_duplicates()
KEYGENE = KEYDF[['gene_symbol', 'pos', 'wt_aa', 'mut_aa']].drop_duplicates()


def match_chunk(ch):
    """Rows of one chunk corresponding to one of our variants."""
    ch = ch.copy()
    ch['_row'] = np.arange(len(ch))
    ch['_ref'] = ch['aaref'].astype(str).str.split(';').str[0]
    ch['_alt'] = ch['aaalt'].astype(str).str.split(';').str[0]

    acc_l = ch['Uniprot_acc'].astype(str).str.split(';')
    pos_l = ch['aapos'].astype(str).str.split(';')
    n = acc_l.str.len().eq(pos_l.str.len())
    rows = set()
    if n.any():
        ex = pd.DataFrame({
            '_row': np.repeat(ch['_row'].values[n.values],
                              acc_l[n].str.len().values),
            'acc': np.concatenate(acc_l[n].values),
            'pos': np.concatenate(pos_l[n].values)})
        ex['pos'] = pd.to_numeric(ex['pos'], errors='coerce')
        ex = ex.dropna(subset=['pos'])
        if len(ex):
            ex['pos'] = ex['pos'].astype('int64')
            ex = ex.merge(ch[['_row', '_ref', '_alt']], on='_row', how='left')
            hit = ex.merge(KEYACC, left_on=['acc', 'pos', '_ref', '_alt'],
                           right_on=['acc', 'pos', 'wt_aa', 'mut_aa'],
                           how='inner')
            rows |= set(hit['_row'])

    miss = ch[~ch['_row'].isin(rows)]              # gene-symbol fallback
    if len(miss):
        g_l = miss['genename'].astype(str).str.split(';')
        p_l = miss['aapos'].astype(str).str.split(';')
        m = g_l.str.len().eq(p_l.str.len())
        if m.any():
            gx = pd.DataFrame({
                '_row': np.repeat(miss['_row'].values[m.values],
                                  g_l[m].str.len().values),
                'gene_symbol': np.concatenate(g_l[m].values),
                'pos': np.concatenate(p_l[m].values)})
            gx['pos'] = pd.to_numeric(gx['pos'], errors='coerce')
            gx = gx.dropna(subset=['pos'])
            if len(gx):
                gx['pos'] = gx['pos'].astype('int64')
                gx = gx.merge(miss[['_row', '_ref', '_alt']], on='_row',
                              how='left')
                gh = gx.merge(KEYGENE,
                              left_on=['gene_symbol', 'pos', '_ref', '_alt'],
                              right_on=['gene_symbol', 'pos', 'wt_aa', 'mut_aa'],
                              how='inner')
                rows |= set(gh['_row'])

    return ch[ch['_row'].isin(rows)].drop(columns=['_row', '_ref', '_alt'])


TAG = {m['name']: os.path.basename(m['name']).replace('.gz', '') for m in MEMBERS}
SIZES = {TAG[m['name']]: m['csize'] for m in MEMBERS}
TOTAL = sum(SIZES.values())


def state_path(tag):
    return f'{OUT}/dbnsfp__{tag}.state.json'


def scan_member(m, tag):
    """Stream one chromosome over a single sustained connection.

    Partial results are checkpointed every PART_EVERY chunks. Resuming re-reads
    the chromosome from its start — gzip has no index, so the bytes must be
    decompressed again either way. What resuming saves is the matched rows
    already found, not the transfer.
    """
    st = {'chunks': 0}
    if os.path.exists(state_path(tag)):
        st = json.load(open(state_path(tag)))
        print(f'  resuming after {st["chunks"]} chunks', flush=True)

    buf, resp = member_stream(m)
    seen, buffered = 0, []
    t0 = time.time()
    try:
        gz = gzip.GzipFile(fileobj=buf)
        reader = pd.read_csv(gz, sep='\t', usecols=USE, dtype=str,
                             na_filter=False, chunksize=CHUNK,
                             on_bad_lines='skip', low_memory=False)
        for i, ch in enumerate(reader):
            seen += len(ch)
            if i < st['chunks']:
                # Already matched in an earlier run. The bytes still have to be
                # decompressed to get here - gzip has no index - so this stretch
                # is silent unless it says so, and on a long resume that looks
                # exactly like a hang.
                if (i + 1) % 5 == 0:
                    print(f'    {tag}  re-reading {i+1}/{st["chunks"]} chunks '
                          f'({seen:,} rows)  {(time.time()-t0)/60:.1f} min',
                          flush=True)
                continue
            got = match_chunk(ch)
            if len(got):
                buffered.append(got)

            if (i + 1) % PART_EVERY == 0 and buffered:
                k = len(glob.glob(f'{OUT}/dbnsfp__{tag}.part*.parquet'))
                pd.concat(buffered, ignore_index=True).to_parquet(
                    f'{OUT}/dbnsfp__{tag}.part{k}.parquet', index=False)
                buffered = []
                json.dump({'chunks': i + 1}, open(state_path(tag), 'w'))

            if (i + 1) % REPORT_EVERY == 0:
                el = time.time() - t0
                kept = sum(len(pd.read_parquet(p)) for p in
                           glob.glob(f'{OUT}/dbnsfp__{tag}.part*.parquet')) \
                    if (i + 1) % (PART_EVERY * 4) == 0 else -1
                k = f'{kept:,}' if kept >= 0 else '...'
                print(f'    {tag}  {seen:>11,} rows  kept {k:>9}  '
                      f'{seen/el:>7,.0f} rows/s  {el/60:>5.1f} min', flush=True)
    finally:
        resp.close()

    if buffered:
        k = len(glob.glob(f'{OUT}/dbnsfp__{tag}.part*.parquet'))
        pd.concat(buffered, ignore_index=True).to_parquet(
            f'{OUT}/dbnsfp__{tag}.part{k}.parquet', index=False)

    got = [pd.read_parquet(p) for p in
           sorted(glob.glob(f'{OUT}/dbnsfp__{tag}.part*.parquet'))]
    out = pd.concat(got, ignore_index=True) if got else pd.DataFrame(columns=USE)
    return out, seen


done = {os.path.basename(f).split('__')[1].replace('.parquet', '')
        for f in glob.glob(f'{OUT}/dbnsfp__*.parquet')
        if '.part' not in f}
print(f'{len(done)} of {len(MEMBERS)} chromosomes complete')
print(f'{TOTAL/1e9:.1f} GB compressed to stream\n', flush=True)

bytes_done = sum(SIZES.get(t, 0) for t in done)
run0 = time.time()

for m in MEMBERS:
    tag = TAG[m['name']]
    if tag in done:
        print(f'skip {tag}', flush=True)
        continue

    print(f'\n=== {tag}  ({m["csize"]/1e9:.1f} GB)  '
          f'overall {100*bytes_done/TOTAL:.0f}% ===', flush=True)
    t0 = time.time()
    df, seen = scan_member(m, tag)
    df.to_parquet(f'{OUT}/dbnsfp__{tag}.parquet', index=False)

    for p in glob.glob(f'{OUT}/dbnsfp__{tag}.part*.parquet'):
        os.remove(p)
    if os.path.exists(state_path(tag)):
        os.remove(state_path(tag))

    bytes_done += m['csize']
    el = time.time() - run0
    eta = el * (TOTAL - bytes_done) / max(bytes_done, 1) / 60
    print(f'{tag}: {seen:,} scanned -> {len(df):,} kept  '
          f'({(time.time()-t0)/60:.1f} min)   ETA {eta:.0f} min', flush=True)

    if len(df) == 0:
        print('  *** WARNING: zero rows kept. Stop and check the join keys '
              'before letting the rest run. ***', flush=True)

Assemble once every chromosome is done.

In [ ]:
import glob
parts = sorted(glob.glob(f'{OUT}/dbnsfp__*.parquet'))
parts = [p for p in parts if '.part' not in p]
full = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
print(f'{len(full):,} rows from {len(parts)} chromosomes')

for c in SCORES:                                   # ';'-separated -> first value
    full[c] = pd.to_numeric(
        full[c].astype(str).str.split(';').str[0].replace({'.': None, '': None}),
        errors='coerce')

cover = full[SCORES].notna().mean().sort_values(ascending=False)
print(f'\ncoverage of our {len(keys):,} variants, best 15 predictors:')
for c, v in cover.head(15).items():
    print(f'  {c:<34} {v:6.1%}')
print(f'\npredictors covering >=50%: {(cover >= 0.5).sum()} of {len(SCORES)}')

full.to_parquet(f'{OUT}/dbnsfp_scores.parquet', index=False)
print(f'\nwrote {OUT}/dbnsfp_scores.parquet '
      f'({os.path.getsize(f"{OUT}/dbnsfp_scores.parquet")/1e6:.0f} MB)')
print('download it and drop it in data/rebuild/')